In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

print(os.listdir("/content/drive/MyDrive"))

Mounted at /content/drive
['patna laboratory.gsheet', 'Untitled spreadsheet.gsheet', 'aadhar and bijli bill.pdf', 'Email List.gsheet', 'html data.gsheet', 'Colab Notebooks', 'Dataset of groundnut plant leaf images for classification and detection', 'datasetnit', 'sorted_dataset', 'main_dataset']


In [ ]:
print(os.listdir("/content/drive/MyDrive/main_dataset"))

['dataset_sir']


In [ ]:
base_path = "/content/drive/MyDrive/main_dataset/dataset_sir/main_dataset"

In [ ]:
import os

print(os.listdir("/content/drive/MyDrive/main_dataset"))
print(os.listdir("/content/drive/MyDrive/main_dataset/dataset_sir"))
print(os.listdir("/content/drive/MyDrive/main_dataset/dataset_sir/main_dataset"))
print(os.listdir("/content/drive/MyDrive/main_dataset/dataset_sir/main_dataset/Train"))

['dataset_sir']
['main_dataset.zip', 'main_dataset', 'YOLOv5_processed_dataset']
['label_num_to_disease_map.json', 'Test', 'Train', 'sorted_dataset']
['Train.csv', 'Train_Images']


In [ ]:
data_path = "/content/drive/MyDrive/main_dataset/dataset_sir/main_dataset/sorted_dataset"

In [ ]:
print(df['Labels'].value_counts())

Labels
3    800
0    800
2    800
4    800
1    800
Name: count, dtype: int64


In [ ]:
import os
import pandas as pd
import shutil
import json

base_path = "/content/drive/MyDrive/main_dataset/dataset_sir/main_dataset"

csv_path = os.path.join(base_path, "Train", "Train.csv")
image_folder = os.path.join(base_path, "Train", "Train_Images")
output_folder = os.path.join(base_path, "sorted_dataset")

df = pd.read_csv(csv_path)

with open(os.path.join(base_path, "label_num_to_disease_map.json")) as f:
    label_map = json.load(f)

# Create folders
for key in label_map:
    os.makedirs(os.path.join(output_folder, label_map[key]), exist_ok=True)

copied = 0

for _, row in df.iterrows():
    img_name = str(row['ID'])
    label = str(row['Labels'])

    src = os.path.join(image_folder, img_name)

    if os.path.exists(src):
        class_name = label_map[label]
        dst = os.path.join(output_folder, class_name, img_name)

        shutil.copy(src, dst)
        copied += 1

print("✅ Total copied:", copied)

✅ Total copied: 4000


In [ ]:
import os

base = output_folder

for cls in os.listdir(base):
    print(cls, ":", len(os.listdir(os.path.join(base, cls))))

CBB : 800
CBSD : 800
CGM : 800
CMD : 800
Healthy : 800


Install TF Hub

In [ ]:
!pip install tensorflow-hub

Import Libraries

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub

In [ ]:
import tensorflow as tf

train_data = tf.keras.preprocessing.image_dataset_from_directory(
    data_path,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(224,224),
    batch_size=32
)

Found 4000 files belonging to 5 classes.
Using 3200 files for training.


Load Validation Data


In [ ]:
val_data = tf.keras.preprocessing.image_dataset_from_directory(
    data_path,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(224,224),
    batch_size=32
)

Found 4000 files belonging to 5 classes.
Using 800 files for validation.


Check Classes

In [ ]:
class_names = train_data.class_names
print("Classes:", class_names)

Classes: ['CBB', 'CBSD', 'CGM', 'CMD', 'Healthy']


Optimize pipeline

In [ ]:
train_data = train_data.cache().shuffle(1000).prefetch(tf.data.AUTOTUNE)
val_data = val_data.cache().prefetch(tf.data.AUTOTUNE)

Build Model

In [ ]:
from tensorflow.keras import layers

preprocess = tf.keras.applications.mobilenet_v3.preprocess_input

base_model = tf.keras.applications.MobileNetV3Large(
    input_shape=(224,224,3),
    include_top=False,
    weights="imagenet"
)

#  Freeze only first 70% layers
for layer in base_model.layers[:int(len(base_model.layers)*0.7)]:
    layer.trainable = False

model = tf.keras.Sequential([
    layers.Input(shape=(224,224,3)),
    layers.Lambda(preprocess),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(5, activation='softmax')
])

model summary

In [ ]:
model.summary()

Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lambda_1 (Lambda)               │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MobileNetV3Large (Functional)   │ (None, 7, 7, 960)      │     2,996,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 960)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 960)            │         3,840 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 128)            │       123,008 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,123,845 (11.92 MB)

 Trainable params: 2,607,157 (9.95 MB)

 Non-trainable params: 516,688 (1.97 MB)

compile

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),  # 🔥 balanced LR
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

Train

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(patience=3)
]

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=20,
    callbacks=callbacks
)

Epoch 1/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 236s 2s/step - accuracy: 0.3325 - loss: 1.9867 - val_accuracy: 0.3425 - val_loss: 2.0583 - learning_rate: 1.0000e-04
Epoch 2/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 236s 2s/step - accuracy: 0.5512 - loss: 1.2017 - val_accuracy: 0.3787 - val_loss: 1.9222 - learning_rate: 1.0000e-04
Epoch 3/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 194s 2s/step - accuracy: 0.6562 - loss: 0.8975 - val_accuracy: 0.4162 - val_loss: 1.7698 - learning_rate: 1.0000e-04
Epoch 4/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 200s 2s/step - accuracy: 0.7341 - loss: 0.6855 - val_accuracy: 0.4363 - val_loss: 1.7791 - learning_rate: 1.0000e-04
Epoch 5/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 191s 2s/step - accuracy: 0.8247 - loss: 0.4826 - val_accuracy: 0.4975 - val_loss: 1.6133 - learning_rate: 1.0000e-04
Epoch 6/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 220s 2s/step - accuracy: 0.8709 - loss: 0.3822 - val_accuracy: 0.5063 - val_loss: 1.5886 - learning_rate: 1.0000e-04
Epoch 7/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 208s 2s/step - acc